In [1]:
# 1) Imports + config

from pathlib import Path
import numpy as np
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, f1_score
from scipy.sparse import hstack, csr_matrix

from xgboost import XGBClassifier

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [2]:
# 2) Load processed dataset v2

PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

processed_path = DATA_PROCESSED / "jailbreak_benchmarks_processed_v2.csv"
df = pd.read_csv(processed_path)

print("Rows:", len(df))
print("\nSplit counts:")
print(df["split"].value_counts())
print("\nLabel counts:")
print(df["label"].value_counts())

Rows: 1647

Split counts:
split
ood_test    768
train       615
val         132
test        132
Name: count, dtype: int64

Label counts:
label
1    993
0    654
Name: count, dtype: int64


In [3]:
# Split views

df_train = df[df["split"] == "train"].copy()
df_val   = df[df["split"] == "val"].copy()
df_test  = df[df["split"] == "test"].copy()
df_ood   = df[df["split"] == "ood_test"].copy()

for name, d in [("train", df_train), ("val", df_val), ("test", df_test), ("ood_test", df_ood)]:
    print(f"{name:8s}", d.shape, d["label"].value_counts().to_dict())


train    (615, 8) {1: 426, 0: 189}
val      (132, 8) {1: 92, 0: 40}
test     (132, 8) {1: 91, 0: 41}
ood_test (768, 8) {1: 384, 0: 384}


In [4]:
# 3) IBVS v1 (same as your 03 notebook)

STRUCTURAL_PATTERNS = {
    "role_redefine": [
        "you are now",
        "from now on you will",
        "pretend to be",
        "act as",
    ],
    "ignore_boundary": [
        "ignore the above",
        "ignore previous instructions",
        "disregard all prior rules",
    ],
    "system_spoof": [
        "<system>",
        "[system]",
        "system:",
    ],
    "meta_override": [
        "as the system prompt",
        "override all policies",
    ],
}

IMPERATIVE_VERBS = [
    "write", "create", "develop", "design", "explain",
    "generate", "produce", "summarise", "summarize",
    "give", "show", "provide", "list", "teach"
]

HARMFUL_ACTION_VERBS = [
    "bypass", "exploit", "manufacture", "build",
    "construct", "smuggle", "hack", "steal",
    "traffic", "fabricate", "weapon", "poison"
]

INSTRUCTION_PATTERNS = [
    r"step[- ]?by[- ]?step",
    r"\bfirst\b",
    r"\bnext\b",
    r"\bthen\b",
    r"\bfinally\b"
]

In [5]:

def ibvs_v1(text: str) -> float:
    """
    Instruction Boundary Violation Score v1
    - Structural overrides (role redefinition, ignoring boundaries, system spoofing, meta override)
    - Imperative starts
    - Harmful action verbs
    - Multi-step instructional structure
    Normalised by token length.
    """
    if not isinstance(text, str):
        return 0.0

    t = text.lower()
    score = 0.0

    # Explicit structural overrides
    if any(p in t for p in STRUCTURAL_PATTERNS["role_redefine"]):
        score += 2.0
    if any(p in t for p in STRUCTURAL_PATTERNS["ignore_boundary"]):
        score += 2.0
    if any(p in t for p in STRUCTURAL_PATTERNS["system_spoof"]):
        score += 2.0
    if any(p in t for p in STRUCTURAL_PATTERNS["meta_override"]):
        score += 2.0

    # Imperative first word
    tokens = t.split()
    first_word = tokens[0] if len(tokens) > 0 else ""
    if first_word in IMPERATIVE_VERBS:
        score += 1.0

    # Harmful action verbs
    if any(v in t for v in HARMFUL_ACTION_VERBS):
        score += 1.5

    # Multi-step cues
    if any(re.search(p, t) for p in INSTRUCTION_PATTERNS):
        score += 1.0

    # Instructional meta-words
    for cue in ["guide", "instructions", "tutorial", "manual"]:
        if cue in t:
            score += 1.0
            break

    length = max(len(tokens), 1)
    return score / length

# Attach IBVS to each split (do it split-wise to avoid accidental leakage patterns)
for split_df in [df_train, df_val, df_test, df_ood]:
    split_df["ibvs_v1"] = split_df["prompt_text"].apply(ibvs_v1)

print("\nIBVS by label (train only, sanity check):")
print(df_train.groupby("label")["ibvs_v1"].describe())


IBVS by label (train only, sanity check):
       count      mean       std  min       25%  50%       75%  max
label                                                              
0      189.0  0.035565  0.061114  0.0  0.000000  0.0  0.076923  0.3
1      426.0  0.130444  0.092360  0.0  0.071429  0.1  0.181818  0.5


In [6]:
# 4) TF–IDF features (fit train only)

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
)

tfidf.fit(df_train["prompt_text"])

X_lex_train = tfidf.transform(df_train["prompt_text"])
X_lex_val   = tfidf.transform(df_val["prompt_text"])
X_lex_test  = tfidf.transform(df_test["prompt_text"])
X_lex_ood   = tfidf.transform(df_ood["prompt_text"])

y_train = df_train["label"].values
y_val   = df_val["label"].values
y_test  = df_test["label"].values
y_ood   = df_ood["label"].values

print("\nTF–IDF shapes:", X_lex_train.shape, X_lex_val.shape, X_lex_test.shape, X_lex_ood.shape)



TF–IDF shapes: (615, 1505) (132, 1505) (132, 1505) (768, 1505)


In [7]:
# 5) Lexical flags (same as the 03 notebook)

OVERRIDE_PATTERNS = [
    r"ignore (all )?(previous|prior) instructions",
    r"you are now",
    r"disregard (the )?(previous|above) rules",
    r"as an unfiltered model",
    r"system prompt",
    r"from now on, you must",
]

def lexical_flags(text: str) -> dict:
    if not isinstance(text, str):
        text = ""
    t = text.lower()
    return {
        "has_ignore_prev": bool(re.search(OVERRIDE_PATTERNS[0], t)),
        "has_you_are_now": "you are now" in t,
        "has_disregard": "disregard" in t and "instructions" in t,
        "has_system_prompt": "system prompt" in t,
        "len_chars": len(text),
        "len_tokens_approx": len(text.split()),
    }

In [8]:
lex_flags_train = pd.DataFrame([lexical_flags(t) for t in df_train["prompt_text"]])
lex_flags_val   = pd.DataFrame([lexical_flags(t) for t in df_val["prompt_text"]])
lex_flags_test  = pd.DataFrame([lexical_flags(t) for t in df_test["prompt_text"]])
lex_flags_ood   = pd.DataFrame([lexical_flags(t) for t in df_ood["prompt_text"]])

# Add IBVS column as an explicit structural feature
lex_flags_train["ibvs_v1"] = df_train["ibvs_v1"].values
lex_flags_val["ibvs_v1"]   = df_val["ibvs_v1"].values
lex_flags_test["ibvs_v1"]  = df_test["ibvs_v1"].values
lex_flags_ood["ibvs_v1"]   = df_ood["ibvs_v1"].values

# Sparse matrices
X_struct_train_all = csr_matrix(lex_flags_train.values.astype(float))
X_struct_val_all   = csr_matrix(lex_flags_val.values.astype(float))
X_struct_test_all  = csr_matrix(lex_flags_test.values.astype(float))
X_struct_ood_all   = csr_matrix(lex_flags_ood.values.astype(float))

# Identify columns (helpful for ablation slicing)
struct_cols = list(lex_flags_train.columns)
print("\nStructural columns:", struct_cols)
# Expected order: 6 flags + ibvs_v1 at the end

# Build "flags only" (no IBVS) by dropping the ibvs column
ibvs_col_idx = struct_cols.index("ibvs_v1")
flag_col_indices = [i for i in range(len(struct_cols)) if i != ibvs_col_idx]

X_struct_train_flags = X_struct_train_all[:, flag_col_indices]
X_struct_val_flags   = X_struct_val_all[:, flag_col_indices]
X_struct_test_flags  = X_struct_test_all[:, flag_col_indices]
X_struct_ood_flags   = X_struct_ood_all[:, flag_col_indices]

print("Struct (flags-only) shape:", X_struct_train_flags.shape)
print("Struct (flags+IBVS) shape:", X_struct_train_all.shape)


Structural columns: ['has_ignore_prev', 'has_you_are_now', 'has_disregard', 'has_system_prompt', 'len_chars', 'len_tokens_approx', 'ibvs_v1']
Struct (flags-only) shape: (615, 6)
Struct (flags+IBVS) shape: (615, 7)


In [9]:
# 6) Model training helpers

def make_xgb():
    # Keep hyperparams fixed across ablations for a fair comparison
    return XGBClassifier(
        objective="binary:logistic",
        n_estimators=400,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=RANDOM_SEED,
    )

def eval_report(model, name, X, y):
    y_pred = model.predict(X)
    print(f"\n=== {name} ===")
    print(classification_report(y, y_pred, digits=3))
    return {
        "accuracy": float(accuracy_score(y, y_pred)),
        "macro_f1": float(f1_score(y, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y, y_pred, average="weighted")),
    }

def run_experiment(exp_name, X_train, X_val, X_test, X_ood):
    model = make_xgb()
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    val_metrics  = eval_report(model, f"{exp_name} | VAL",  X_val,  y_val)
    test_metrics = eval_report(model, f"{exp_name} | TEST", X_test, y_test)
    ood_metrics  = eval_report(model, f"{exp_name} | OOD",  X_ood,  y_ood)

    out = {
        "model": exp_name,
        "val_acc": val_metrics["accuracy"],
        "test_acc": test_metrics["accuracy"],
        "ood_acc": ood_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "test_macro_f1": test_metrics["macro_f1"],
        "ood_macro_f1": ood_metrics["macro_f1"],
    }
    return out


In [10]:
# 7) Ablations (M1 / M2 / M3)

results = []

# M1: TF–IDF only
results.append(
    run_experiment(
        "M1_TFIDF_ONLY",
        X_lex_train,
        X_lex_val,
        X_lex_test,
        X_lex_ood,
    )
)

# M2: TF–IDF + flags (no IBVS)
X_train_m2 = hstack([X_lex_train, X_struct_train_flags]).tocsr()
X_val_m2   = hstack([X_lex_val,   X_struct_val_flags]).tocsr()
X_test_m2  = hstack([X_lex_test,  X_struct_test_flags]).tocsr()
X_ood_m2   = hstack([X_lex_ood,   X_struct_ood_flags]).tocsr()

results.append(
    run_experiment(
        "M2_TFIDF_PLUS_FLAGS",
        X_train_m2,
        X_val_m2,
        X_test_m2,
        X_ood_m2,
    )
)

# M3: TF–IDF + flags + IBVS (full structural)
X_train_m3 = hstack([X_lex_train, X_struct_train_all]).tocsr()
X_val_m3   = hstack([X_lex_val,   X_struct_val_all]).tocsr()
X_test_m3  = hstack([X_lex_test,  X_struct_test_all]).tocsr()
X_ood_m3   = hstack([X_lex_ood,   X_struct_ood_all]).tocsr()

results.append(
    run_experiment(
        "M3_TFIDF_PLUS_FLAGS_PLUS_IBVS",
        X_train_m3,
        X_val_m3,
        X_test_m3,
        X_ood_m3,
    )
)


=== M1_TFIDF_ONLY | VAL ===
              precision    recall  f1-score   support

           0      0.854     0.875     0.864        40
           1      0.945     0.935     0.940        92

    accuracy                          0.917       132
   macro avg      0.899     0.905     0.902       132
weighted avg      0.917     0.917     0.917       132


=== M1_TFIDF_ONLY | TEST ===
              precision    recall  f1-score   support

           0      0.811     0.732     0.769        41
           1      0.884     0.923     0.903        91

    accuracy                          0.864       132
   macro avg      0.848     0.827     0.836       132
weighted avg      0.861     0.864     0.862       132


=== M1_TFIDF_ONLY | OOD ===
              precision    recall  f1-score   support

           0      0.646     0.823     0.724       384
           1      0.756     0.549     0.637       384

    accuracy                          0.686       768
   macro avg      0.701     0.686     0.

In [11]:
# 8) Results table (copy into dissertation)

results_df = pd.DataFrame(results)

# Sort by OOD accuracy descending (usually most important for your story)
results_df = results_df.sort_values(by="ood_acc", ascending=False).reset_index(drop=True)

# Round for readability
display_cols = [
    "model",
    "val_acc", "test_acc", "ood_acc",
    "val_macro_f1", "test_macro_f1", "ood_macro_f1"
]
results_display = results_df[display_cols].copy()
for c in display_cols[1:]:
    results_display[c] = results_display[c].map(lambda x: round(x, 4))

print("\n=== Ablation Summary (v2) ===")
display(results_display)


=== Ablation Summary (v2) ===


,model,val_acc,test_acc,ood_acc,val_macro_f1,test_macro_f1,ood_macro_f1
0,M2_TFIDF_PLUS_FLAGS,0.9167,0.8636,0.7122,0.8992,0.8310,0.7105
1,M3_TFIDF_PLUS_FLAGS_PLUS_IBVS,0.8864,0.8636,0.6875,0.8682,0.8408,0.6855
2,M1_TFIDF_ONLY,0.9167,0.8636,0.6862,0.9020,0.8362,0.6802
